# DermaLens AI · Real-Image Quality Audit, Duplicate Grouping & Frozen 70/15/15 Split

**Kaggle execution notebook — independent of GitHub; no repository writes.**

**Input:** `shanmukhadattaboda/facial-skin-conditions-real-image-collection`, Version 2 (6 class folders).  
**Outputs:** `/kaggle/working/dermalens_real_split/` (audit CSVs + manifests) and `/kaggle/working/dermalens_real_70_15_15.zip` (`train/`, `validation/`, `test/`, plus `reports/`).

**Read before running:** This notebook groups exact file / decoded-pixel duplicates, 4-orientation pHash duplicates (Hamming ≤ 5) and high-similarity ResNet-50 ImageNet embedding pairs. All groups are kept within one split. **Near-duplicate detection is heuristic, not a patient-ID guarantee.** Human review of suspicious clusters, source labels, and licensing remains required. It **does not alter source photographs**; model-specific resizing / normalization and train-only augmentation belong in training notebooks.

**Kaggle settings:** GPU T4 recommended; enable Internet in Notebook Settings on first run to fetch ImageNet ResNet-50 weights, unless already cached. Add your dataset via `+ Add Input`. Total RAM ~16 GB+ recommended. Run top-to-bottom. Every main stage writes CSV/JSON snapshots and prints counts. If execution stops, inspect the last numbered cell and `reports/`.

**Safeguards:** decode errors, predominantly blank images and cross-class duplicates are quarantined; embedding edges do *not* by themselves assign diagnoses. Similarity can join distinct people — manually inspect large groups. The final ZIP preserves the source image bytes and adds unique names to avoid filename collisions.


In [ ]:
# Cell 01 — Environment & dependencies (run once)
import sys, subprocess, importlib.util
for module, pip_name in [('imagehash','ImageHash'), ('sklearn','scikit-learn')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
import os, gc, re, json, time, hashlib, zipfile, warnings, collections
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageStat, UnidentifiedImageError
import imagehash
from sklearn.model_selection import GroupShuffleSplit
import torch
from torchvision import models, transforms
print('Python:',sys.version.split()[0], 'Torch:',torch.__version__, 'GPU:',torch.cuda.is_available())
print('NumPy:',np.__version__, 'pandas:',pd.__version__)


## Step 1 — Set paths and safety thresholds
**Class folder names exactly match the supplied Kaggle dataset.** All output paths are under `/kaggle/working` (writable). Set `REVIEW_ONLY=True` to inspect candidate groups without generating a ZIP. **The notebook never deletes or edits input files.**

In [ ]:
# Cell 02 — User-configurable settings
DATA_ROOT = Path('/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection/Face_Skin_datset')
# Some Kaggle input mounts use a shorter path; safely detect that case:
if not DATA_ROOT.is_dir():
    options = list(Path('/kaggle/input').glob('**/Face_Skin_datset'))
    if len(options) == 1: DATA_ROOT = options[0]
assert DATA_ROOT.is_dir(), f'Missing dataset root: {DATA_ROOT}. Use Add Input and check the folder name.'
OUT = Path('/kaggle/working/dermalens_real_split'); OUT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path('/kaggle/working/dermalens_real_70_15_15.zip')
CLASSES = ['Acne', 'Fungal_infection', 'Hyper_Pigmentation', 'Normal_Skin', 'Skin_cancer', 'Vitiligo']
EXTENSIONS = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
SEED = 42; PHASH_THRESHOLD = 5; PHASH_SIZE = 8
EMBED_START = 0.90; EMBED_STEP = 0.02; EMBED_MAX = 0.98
MAX_GROUP_SHARE = 0.02; MAX_EMBED_EDGES = 2_000_000
GROUP_SPLIT_TRIES = 200; BATCH_SIZE = 32; SIM_CHUNK = 256
MIN_EDGE = 64                    # smaller images flagged for review, not auto-removed
BLANK_STD_THRESHOLD = 4.0       # very low-variance RGB images quarantined
REVIEW_ONLY = False
MANUAL_APPROVAL_FOR_LARGE_GROUPS = False  # set True to require manual group review gate
assert all((DATA_ROOT / cls).is_dir() for cls in CLASSES), 'One or more six class directories missing.'
print('Dataset:', DATA_ROOT)
print('Output:', OUT)
print('Classes:', CLASSES)


In [ ]:
# Cell 03 — Reproducibility / run settings snapshot
np.random.seed(SEED); torch.manual_seed(SEED)
config = {'dataset_root':str(DATA_ROOT),'class_folders':CLASSES, 'seed':SEED,
          'phash_size':PHASH_SIZE,'phash_threshold':PHASH_THRESHOLD,
          'embedding_model':'torchvision_resnet50_IMAGENET1K_V2_avgpool',
          'embed_threshold_start':EMBED_START,'embed_step':EMBED_STEP,'embed_max':EMBED_MAX,
          'max_group_share':MAX_GROUP_SHARE,'max_embedding_edges':MAX_EMBED_EDGES,
          'group_split_tries':GROUP_SPLIT_TRIES,'min_edge_warning':MIN_EDGE,
          'blank_std_threshold':BLANK_STD_THRESHOLD,
          'generated_at_utc':datetime.now(timezone.utc).isoformat()}
(OUT/'run_config.json').write_text(json.dumps(config,indent=2))
print(json.dumps(config,indent=2))


## Step 2 — Source inventory & image integrity checks
Scan all images; record resolution, modes, sizes, **file SHA-256**, **decoded RGB pixel SHA-256**, very dark/blank warnings, and read errors. `pixel_sha256` is used to link images that are pixel-identical despite different file encodings. **Small images are flagged but retained**. No cropping/resizing before duplicate detection. Wrong labels cannot be reliably identified automatically and need manual audits.

In [ ]:
# Cell 04 — Enumerate files
rows=[]; other=[]
for cls in CLASSES:
    for p in sorted((DATA_ROOT/cls).rglob('*')):
        if p.is_file():
            if p.suffix.lower() in EXTENSIONS:
                rows.append({'path':str(p),'relative_path':p.relative_to(DATA_ROOT).as_posix(), 'class':cls})
            else: other.append({'path':str(p),'class':cls,'suffix':p.suffix.lower()})
files=pd.DataFrame(rows)
assert len(files)>0,'No recognized images found.'
files.insert(0,'image_id',np.arange(len(files),dtype=int))
files.to_csv(OUT/'00_source_inventory.csv',index=False)
pd.DataFrame(other,columns=['path','class','suffix']).to_csv(OUT/'00_ignored_nonimages.csv',index=False)
print('Images discovered:',len(files),'Non-image files ignored:',len(other))
print(files.groupby('class').size().reindex(CLASSES,fill_value=0).to_string())
display(files.head(5))


In [ ]:
# Cell 05 — Full integrity/hash scan; one line per file
from tqdm.auto import tqdm

def scan_image(row):
    p=Path(row.path)
    r={'image_id':row.image_id, 'path':str(p),'relative_path':row.relative_path,
       'class':row['class'], 'bytes':p.stat().st_size,'status':'ok', 'reason':'',
       'width':np.nan,'height':np.nan,'format':'','pixel_sha256':'', 'sha256':''}
    try:
        h=hashlib.sha256()
        with p.open('rb') as f:
            for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
        r['sha256']=h.hexdigest()
        with Image.open(p) as im:
            r['format']=str(im.format)
            rgb=ImageOps.exif_transpose(im).convert('RGB')
            w,he=rgb.size; r['width']=w; r['height']=he
            # Content hash includes dimensions, so different shaped bitmaps cannot collide semantically.
            r['pixel_sha256']=hashlib.sha256(f'{w}x{he}:'.encode()+rgb.tobytes()).hexdigest()
            thumb=rgb.copy(); thumb.thumbnail((256,256))
            std=float(np.mean(ImageStat.Stat(thumb).stddev))
            r['rgb_std_mean']=round(std,4)
            if w<MIN_EDGE or he<MIN_EDGE: r['reason']='small_image_review'
            if std < BLANK_STD_THRESHOLD:
                r['status']='excluded_blank'; r['reason']='near_uniform_image'
    except Exception as exc:
        r['status']='excluded_corrupt';r['reason']=f'{type(exc).__name__}: {str(exc)[:160]}'
    return r

audit=pd.DataFrame([scan_image(row) for _,row in tqdm(files.iterrows(),total=len(files),desc='Integrity and SHA-256')])
audit.to_csv(OUT/'01_quality_and_sha256.csv',index=False)
print(audit.groupby(['class','status']).size().to_string())
print('Image size warnings:',audit.reason.eq('small_image_review').sum())
clean=audit[audit.status=='ok'].copy().reset_index(drop=True)
assert len(clean)>0,'All images excluded. Inspect quality report.'
print('Usable after integrity check:',len(clean),'Excluded:',len(audit)-len(clean))


## Step 3 — Exact and orientation-aware duplicate links
1. Merge byte-identical images (`sha256`) and decoded-pixel-identical images (`pixel_sha256`).
2. Compute 64-bit pHash of original, horizontal mirror, vertical flip, and 180° rotation; compare all image pairs in bounded blocks using a lookup-table bit count. The minimum over oriented comparisons identifies candidates at Hamming ≤ 5.
3. Record edges and per-stage counts. Unlike arbitrary class-only comparisons, **all six categories are compared together** to catch cross-label duplication.

The pHash pass can be CPU-heavy for ~9.8k images. It is deliberately all-pairs, so it does not miss candidates because of approximate nearest-neighbor indexing.

In [ ]:
# Cell 06 — Exact-content duplicate links (keep everything until groups are assembled)
exact_pairs=[]
for col,kind in [('sha256','sha256_bytes'),('pixel_sha256','sha256_pixels')]:
    for _,part in clean.groupby(col,sort=False):
        ids=part.image_id.astype(int).tolist()
        if len(ids)>1: exact_pairs.extend((ids[0],i,kind,1.0) for i in ids[1:])
exact_edges=pd.DataFrame(exact_pairs,columns=['id_a','id_b','method','score'])
exact_edges.to_csv(OUT/'02_exact_duplicate_edges.csv',index=False)
print('Exact-link records:',len(exact_edges),'Unique images in exact links:',len(set(exact_edges.id_a)|set(exact_edges.id_b)) if len(exact_edges) else 0)


In [ ]:
# Cell 07 — Flip-aware pHash signatures

def phash_four(path):
    with Image.open(path) as im:
        rgb=ImageOps.exif_transpose(im).convert('RGB')
        variants=[rgb, ImageOps.mirror(rgb), ImageOps.flip(rgb),rgb.transpose(Image.Transpose.ROTATE_180)]
        return [int(str(imagehash.phash(v,hash_size=PHASH_SIZE)),16) for v in variants]

H=np.array([phash_four(p) for p in tqdm(clean.path,desc='4 x pHash')],dtype=np.uint64)
for k,name in enumerate(['original','horizontal','vertical','rotation180']):
    clean['phash_'+name]=[f'{int(v):016x}' for v in H[:,k]]
clean[['image_id','class','relative_path']+[c for c in clean.columns if c.startswith('phash_')]].to_csv(OUT/'03_perceptual_hashes.csv',index=False)
print('Hash matrix shape:',H.shape)


In [ ]:
# Cell 08 — Exact all-pairs Hamming comparison in bounded blocks
# Includes original-to-each-transformation in both directions. Duplicates from
# rotated/flipped crops may produce Hamming<=5. We keep MIN over these 7 checks.
POPCOUNT=np.array([int(i).bit_count() for i in range(256)],dtype=np.uint8)
def bit_dist(left,right):
    x=np.bitwise_xor(left[:,None],right[None,:]).copy()
    bits=POPCOUNT[x.view(np.uint8).reshape(len(left),len(right),8)].sum(axis=2,dtype=np.uint8)
    return bits

phash_pairs=[]; nimg=len(clean); block=192
for si in tqdm(range(0,nimg,block),desc='All-pairs flip-aware pHash'):
    ei=min(nimg,si+block)
    for sj in range(si,nimg,block):
        ej=min(nimg,sj+block)
        d=bit_dist(H[si:ei,0],H[sj:ej,0])
        for orient in range(1,4):
            np.minimum(d,bit_dist(H[si:ei,0],H[sj:ej,orient]),out=d)
            np.minimum(d,bit_dist(H[si:ei,orient],H[sj:ej,0]),out=d)
        ii,jj=np.where(d<=PHASH_THRESHOLD)
        for u,v in zip(ii,jj):
            a=si+int(u); b=sj+int(v)
            if a < b: phash_pairs.append((int(clean.iloc[a].image_id),int(clean.iloc[b].image_id),int(d[u,v])))
phash_edges=pd.DataFrame(phash_pairs,columns=['id_a','id_b','hamming'])
phash_edges.to_csv(OUT/'04_phash_candidate_edges.csv',index=False)
print('pHash linked pairs:',len(phash_edges))


## Step 4 — ResNet-50 image embeddings (ImageNet-pretrained, **not trained on your split**)
Input images are EXIF-corrected and resized for *feature extraction only*. We extract the 2048-dimensional global-average-pooled ResNet-50 features for original and horizontally mirrored images, normalize them and compute similarities. This pretrained backbone is used only as a fixed visual duplicate detector, **not** as a disease-classification model here.

**If loading pretrained weights fails:** enable Kaggle Internet temporarily, then rerun. The notebook intentionally stops instead of silently using untrained weights.

In [ ]:
# Cell 09 — Load true ImageNet weights and embed both orientations
try:
    weights=models.ResNet50_Weights.IMAGENET1K_V2
    net=models.resnet50(weights=weights)
except Exception as e:
    raise RuntimeError('Could not load pretrained ResNet50. Enable Kaggle Internet or cache torchvision ImageNet weights. DO NOT substitute random weights.') from e
net.fc=torch.nn.Identity(); net.eval()
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); net=net.to(device)
preprocess=weights.transforms()

def load_pair(p):
    with Image.open(p) as im:
        rgb=ImageOps.exif_transpose(im).convert('RGB')
        return preprocess(rgb),preprocess(ImageOps.mirror(rgb))

orig=[]; mirrored=[]
with torch.inference_mode():
    for s in tqdm(range(0,nimg,BATCH_SIZE),desc='ResNet-50 embeddings'):
        batch=[load_pair(p) for p in clean.path.iloc[s:s+BATCH_SIZE]]
        a=torch.stack([v[0] for v in batch]).to(device)
        b=torch.stack([v[1] for v in batch]).to(device)
        orig.append(net(a).detach().float().cpu())
        mirrored.append(net(b).detach().float().cpu())
E=torch.cat(orig).numpy().astype('float32'); F=torch.cat(mirrored).numpy().astype('float32')
assert E.shape==(nimg,2048) and F.shape==E.shape
np.savez_compressed(OUT/'05_resnet50_embeddings.npz',image_id=clean.image_id.to_numpy(),features=E,mirrored_features=F)
print('Saved embeddings:',E.shape,'device:',device)
del net,orig,mirrored; gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()


In [ ]:
# Cell 10 — Normalized, flip-aware cosine similarity candidate generation
# The original and its mirror are both examined. Cross-class edges are retained
# for conflict review rather than automatically relabeling photographs.
def unit(x):
    x=x.astype('float32',copy=True)
    x-=x.mean(axis=0,keepdims=True)  # mean-centering matches repository group splitter
    x/=np.maximum(np.linalg.norm(x,axis=1,keepdims=True),1e-12)
    return x
# Center both orientations by one common population mean.
mu=np.concatenate([E,F],axis=0).mean(axis=0,keepdims=True)
A=E-mu; B=F-mu
A/=np.maximum(np.linalg.norm(A,axis=1,keepdims=True),1e-12)
B/=np.maximum(np.linalg.norm(B,axis=1,keepdims=True),1e-12)
del E,F; gc.collect()
# GPU accelerates approximate similarity matrix calculations, exact pair selection is still exhaustive.
AA=torch.from_numpy(A).to(device); BB=torch.from_numpy(B).to(device)
embedding_pairs=[]
with torch.inference_mode():
    for start in tqdm(range(0,nimg,SIM_CHUNK),desc='Embedding similarities'):
        end=min(start+SIM_CHUNK,nimg)
        sim=torch.maximum(AA[start:end]@AA.T,AA[start:end]@BB.T)
        sim=torch.maximum(sim,BB[start:end]@AA.T)
        sim=torch.maximum(sim,BB[start:end]@BB.T)
        sim=sim.cpu().numpy()
        r,c=np.where(sim >= EMBED_START)
        valid=c>=(r+start+1);r=r[valid];c=c[valid]
        embedding_pairs.extend((int(clean.iloc[start+int(i)].image_id),int(clean.iloc[int(j)].image_id),float(sim[i,j])) for i,j in zip(r,c))
        if len(embedding_pairs)>MAX_EMBED_EDGES:
            raise RuntimeError('Too many embedding links at 0.90; inspect thresholds and false links before continuing. No ZIP exported.')
embed_edges=pd.DataFrame(embedding_pairs,columns=['id_a','id_b','cosine'])
embed_edges.to_csv(OUT/'06_embedding_candidate_edges.csv',index=False)
print('Embedding candidates >=',EMBED_START,':',len(embed_edges))
del AA,BB,A,B; gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()


## Step 5 — Union-Find connected components & ambiguity audit
All exact and pHash links are always kept. Embedding similarity starts at 0.90; increase the **embedding** threshold by 0.02 only if a connected component exceeds ~2% of the dataset. The group-size limit is a *warning*, not a reason to split an identified duplicate cluster.

A group with **multiple target class labels** is **quarantined in full** until a human resolves the conflict. A suspicious large group is recorded for review; optionally set `MANUAL_APPROVAL_FOR_LARGE_GROUPS=True` to block export.

In [ ]:
# Cell 11 — Union-Find implementation and adaptive grouping
class UnionFind:
    def __init__(self,ids):self.p={int(i):int(i) for i in ids};self.sz={int(i):1 for i in ids}
    def root(self,x):
        x=int(x)
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]];x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.root(a),self.root(b)
        if a==b:return
        if self.sz[a]<self.sz[b]:a,b=b,a
        self.p[b]=a;self.sz[a]+=self.sz[b]

def build_groups(threshold):
    uf=UnionFind(clean.image_id)
    for r in exact_edges.itertuples(index=False):uf.union(r.id_a,r.id_b)
    for r in phash_edges.itertuples(index=False):uf.union(r.id_a,r.id_b)
    for r in embed_edges.itertuples(index=False):
        if r.cosine>=threshold:uf.union(r.id_a,r.id_b)
    roots=pd.Series([uf.root(x) for x in clean.image_id])
    sizes=roots.value_counts()
    return roots.to_numpy(),sizes

threshold_history=[];tau=EMBED_START
while True:
    roots,sizes=build_groups(tau)
    share=sizes.iloc[0]/nimg
    threshold_history.append({'cosine_threshold':round(tau,2),'groups':len(sizes),'largest_group':int(sizes.iloc[0]),'largest_group_share':round(share,5)})
    if share<=MAX_GROUP_SHARE or tau>=EMBED_MAX:break
    tau=round(min(EMBED_MAX,tau+EMBED_STEP),2)

pd.DataFrame(threshold_history).to_csv(OUT/'07_adaptive_threshold_audit.csv',index=False)
clean['group_id']=['RG'+str(x).zfill(6) for x in roots]
summary=(clean.groupby('group_id',as_index=False).agg(group_size=('image_id','size'),
         class_count=('class','nunique'),classes=('class',lambda s:'|'.join(sorted(s.unique()))),
         representative_id=('image_id','min')))
summary['review_flag']=np.where(summary.class_count>1,'CROSS_CLASS_CONFLICT',np.where(summary.group_size>max(10,int(nimg*MAX_GROUP_SHARE)),'LARGE_GROUP_REVIEW',''))
summary.to_csv(OUT/'08_group_summary.csv',index=False)
clean.to_csv(OUT/'08_image_group_assignments.csv',index=False)
print('Chosen embedding threshold:',tau,'Total groups:',len(summary),'Largest group:',summary.group_size.max())
print('Multi-image groups:',int((summary.group_size>1).sum()))
print('Conflicting-label groups:',int((summary.class_count>1).sum()))
print('Groups requiring review:',int((summary.review_flag!='').sum()))
display(summary.sort_values('group_size',ascending=False).head(20))


In [ ]:
# Cell 11b — Optional visual gallery of largest/ambiguous clusters (portable HTML)
import base64, io, html
from IPython.display import HTML
selected=summary.sort_values(['class_count','group_size'],ascending=False).head(30)
blocks=['<html><head><meta charset="utf-8"><style>body{font:14px Arial;margin:18px;}section{border:1px solid #ddd;padding:12px;margin:10px 0;border-radius:8px;}img{height:110px;max-width:155px;object-fit:contain;margin:3px;border:1px solid #eee;}small{color:#666}</style></head><body><h2>DermaLens group review · largest/conflicting components</h2>']
for rec in selected.to_dict('records'):
    member=clean[clean.group_id==rec['group_id']].head(12)
    blocks.append('<section><b>'+html.escape(rec['group_id'])+'</b> | size '+str(rec['group_size'])+' | '+html.escape(rec['classes'])+'<div>')
    for item in member.to_dict('records'):
        try:
            with Image.open(item['path']) as im:
                t=ImageOps.exif_transpose(im).convert('RGB');t.thumbnail((155,110));buf=io.BytesIO();t.save(buf,format='JPEG',quality=65)
            blocks.append('<img title="'+html.escape(item['relative_path'],quote=True)+'" src="data:image/jpeg;base64,'+base64.b64encode(buf.getvalue()).decode()+'"/>')
        except Exception:pass
    blocks.append('</div><small>Sample up to 12 images; consult CSV for complete group.</small></section>')
blocks.append('</body></html>')
gallery_path=OUT/'08_group_review_gallery.html';gallery_path.write_text(''.join(blocks),encoding='utf-8')
print('Saved gallery:',gallery_path,'(open in browser to inspect suspect groups)')


In [ ]:
# Cell 12 — Quarantine cross-class groups and save decisions
conflict_groups=set(summary.loc[summary.class_count>1,'group_id'])
quarantine=clean[clean.group_id.isin(conflict_groups)].copy()
quarantine['quarantine_reason']='cross_class_duplicate_group'
quarantine.to_csv(OUT/'09_quarantined_cross_class_groups.csv',index=False)
eligible=clean[~clean.group_id.isin(conflict_groups)].copy().reset_index(drop=True)
large=summary[summary.review_flag=='LARGE_GROUP_REVIEW'].copy()
large.to_csv(OUT/'09_large_groups_for_manual_review.csv',index=False)
print('Images quarantined:',len(quarantine),'Eligible:',len(eligible),'Large groups flagged:',len(large))
if MANUAL_APPROVAL_FOR_LARGE_GROUPS and len(large):
    raise RuntimeError('Large groups flagged for manual review. Inspect 09_large_groups_for_manual_review.csv first.')
assert len(eligible)>0,'No eligible images.'
print('Note: No automatic disease relabeling or cross-class merging into one class was performed.')


## Step 6 — Freeze class-aware, whole-group 70/15/15 partitions
First select ~30% as held-out, then divide the held-out groups evenly into validation/test. Search 200 seeded trials each stage for class balance and size proximity. **No group can cross splits.** A class with fewer than three distinct groups cannot be safely represented in all three sets: the notebook stops instead of faking a split.

In [ ]:
# Cell 13 — Balanced repeated GroupShuffleSplit
labels={c:i for i,c in enumerate(CLASSES)}
eligible['label']=eligible['class'].map(labels)
assert eligible.label.notna().all()
for cls,g in eligible.groupby('class'):
    assert g.group_id.nunique()>=3, f'{cls}: less than 3 independent groups; cannot guarantee train/val/test.'

def best_group_shuffle(frame, fraction, seed, tries=200):
    y=frame['label'].to_numpy(int); groups=frame['group_id'].to_numpy();
    desired=np.bincount(y,minlength=len(CLASSES))/len(y)
    best=None
    for attempt in range(tries):
        splitter=GroupShuffleSplit(n_splits=1,test_size=fraction,random_state=seed+attempt)
        tr,te=next(splitter.split(frame,y,groups=groups))
        if not len(tr) or not len(te):continue
        counts=np.bincount(y[te],minlength=len(CLASSES))
        if np.any(counts==0) or np.any(np.bincount(y[tr],minlength=len(CLASSES))==0):continue
        prop=counts/len(te)
        score=float(np.max(np.abs(prop-desired)/np.maximum(desired,1e-6)) + 2*abs(len(te)/len(frame)-fraction))
        if best is None or score<best[0]:best=(score,tr,te,attempt)
    if best is None:raise RuntimeError('No balanced whole-group split found; review class/group distribution.')
    return best

score1,tr_pos,hold_pos,trial1=best_group_shuffle(eligible,.30,SEED,GROUP_SPLIT_TRIES)
held=eligible.iloc[hold_pos].reset_index(drop=True)
score2,va_pos,te_pos,trial2=best_group_shuffle(held,.50,SEED+10_000,GROUP_SPLIT_TRIES)
parts={'train':eligible.iloc[tr_pos].copy(),'validation':held.iloc[va_pos].copy(),'test':held.iloc[te_pos].copy()}
manifest=pd.concat([df.assign(split=split) for split,df in parts.items()],ignore_index=True)
manifest.to_csv(OUT/'10_frozen_real_split_manifest.csv',index=False)
print('Selected trials:',trial1,trial2,'Scores:',round(score1,4),round(score2,4))
print('Final eligible image split counts:')
print(manifest.split.value_counts().to_string())
display(pd.crosstab(manifest['class'],manifest['split']).reindex(index=CLASSES,columns=['train','validation','test'],fill_value=0))


In [ ]:
# Cell 14 — Leakage, group coverage and distribution checks
splits=['train','validation','test']
assert not manifest.image_id.duplicated().any()
assert len(manifest)==len(eligible)
assert manifest.groupby('group_id').split.nunique().max()==1,'ERROR: group crosses partitions!'
assert manifest.path.nunique()==len(manifest),'ERROR: path reused!'
assert manifest.groupby('sha256').split.nunique().max()==1,'ERROR: SHA256 leakage!'
assert manifest.groupby('pixel_sha256').split.nunique().max()==1,'ERROR: identical pixel leakage!'
assert all(pd.crosstab(manifest['class'],manifest['split']).reindex(index=CLASSES,columns=splits,fill_value=0).min(axis=1)>0),'ERROR: a class is absent in one split.'
assert not set(quarantine.image_id)&set(manifest.image_id),'ERROR: quarantined image escaped!'
counts=pd.crosstab(manifest['class'],manifest['split']).reindex(index=CLASSES,columns=splits,fill_value=0)
counts['total']=counts.sum(axis=1)
for split in splits:counts[f'{split}_pct']=np.round(counts[split]/counts.total*100,2)
counts.to_csv(OUT/'11_per_class_split_counts.csv')
per_group=(manifest.groupby(['split','class'],as_index=False).group_id.nunique().rename(columns={'group_id':'independent_groups'}))
per_group.to_csv(OUT/'11_group_counts_by_split_class.csv',index=False)
print('PASS: no duplicate path, exact bytes, identical pixels or identified group across splits.')
display(counts)


## Step 7 — Reports, human-readable audit and training-ready ZIP
The ZIP contains `train/<class>/`, `validation/<class>/`, and `test/<class>/`. **Original bytes and file extensions are preserved**; file names are prefixed with an image ID to prevent collisions. `reports/` includes audit CSV files, group metadata and a `README.txt`. No class balancing or training augmentation is performed.

If `REVIEW_ONLY=True`, the ZIP is not created. Users can review audit tables and rerun the notebook before freezing the archive.

In [ ]:
# Cell 15 — Stage-by-stage accounting CSV and JSON
status_counts=audit.status.value_counts()
stages=[
 ('00_source_inventory',len(files),'all discovered images'),
 ('01_integrity_pass',len(clean),'readable and non-blank images'),
 ('02_exact_candidate_edges',len(exact_edges),'exact byte/pixel duplicate links'),
 ('03_phash_candidate_edges',len(phash_edges),'flip / 180-degree pHash links'),
 ('04_resnet_candidate_edges',len(embed_edges),'ImageNet ResNet cosine >= initial threshold'),
 ('05_duplicate_components',len(summary),'all connected components'),
 ('06_multilabel_components',int((summary.class_count>1).sum()),'cross-label components requiring review'),
 ('07_quarantined_images',len(quarantine),'all images in cross-label components'),
 ('08_eligible_images',len(eligible),'eligible after quality and conflict quarantine'),
 ('09_train',len(parts['train']),'training images'),
 ('10_validation',len(parts['validation']),'validation images'),
 ('11_test',len(parts['test']),'test images'),
]
stage_table=pd.DataFrame(stages,columns=['stage','count','description'])
stage_table.to_csv(OUT/'12_pipeline_stage_counts.csv',index=False)
quality_by_class=pd.crosstab(audit['class'],audit['status']).reindex(CLASSES,fill_value=0)
quality_by_class.to_csv(OUT/'12_quality_counts_by_class.csv')
report={'source_image_count':len(files),'quality_excluded':len(files)-len(clean),'quarantined_conflicting':len(quarantine),
        'eligible':len(eligible),'group_count_all':len(summary),'group_count_final':manifest.group_id.nunique(),
        'selected_cosine_tau':tau,'split_counts':manifest.split.value_counts().to_dict(),
        'split_percentages':(manifest.split.value_counts(normalize=True)*100).round(3).to_dict(),
        'seed':SEED,'quality_counts':status_counts.to_dict(),
        'limitations':['No patient IDs: duplicate groups cannot establish patient independence',
                       'Cross-class conflicting groups quarantined; within-class mislabels remain possible',
                       'Visual embedding groups can join different people and require review',
                       'No automated clinical verification or source licensing validation']}
(OUT/'12_final_summary.json').write_text(json.dumps(report,indent=2))
print(stage_table.to_string(index=False))
print('\nSplit percentages:',report['split_percentages'])


In [ ]:
# Cell 16 — Create ZIP with class folders + audit reports, and verify archive counts
README='DermaLens AI real-image frozen dataset\nFolders: train/, validation/, test/; classes as uploaded.\nSplit target: 70/15/15, group-aware; actual sizes in reports/11_per_class_split_counts.csv.\nImages retain original bytes; filenames prefixed with unique image_id.\nQuarantined cross-class groups and corrupt/blank files are EXCLUDED from train/validation/test.\nThese splits are near-duplicate-group separated, NOT guaranteed patient-disjoint.\nDo not alter these partitions across model experiments.\nReview the licensing of each upstream dataset before redistribution.\n'
if REVIEW_ONLY:
    print('REVIEW_ONLY=True: no ZIP created. Inspect reports and rerun with False when satisfied.')
else:
    if ZIP_PATH.exists():ZIP_PATH.unlink()
    with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3,allowZip64=True) as z:
        for rec in tqdm(manifest[['path','image_id','class','split']].to_dict('records'),desc='Write images'):
            p=Path(rec['path'])
            z.write(p,f"{rec['split']}/{rec['class']}/{int(rec['image_id']):06d}_{p.name}")
        z.writestr('reports/README.txt',README)
        for report_path in sorted(OUT.iterdir()):
            if report_path.suffix.lower() in {'.csv','.json','.html'}:
                z.write(report_path,f'reports/{report_path.name}')
    with zipfile.ZipFile(ZIP_PATH,'r') as z:
        names=[x.filename for x in z.infolist() if x.filename.lower().endswith(tuple(EXTENSIONS))]
        actual=collections.Counter((x.split('/')[0],x.split('/')[1]) for x in names)
        expected=collections.Counter((r['split'],r['class']) for r in manifest[['split','class']].to_dict('records'))
        assert actual==expected, 'ZIP folder/class counts do not match split manifest!'
        assert z.testzip() is None, 'ZIP integrity test failed!'
        print('ZIP contents verified:',len(names),'images; reports:',sum(x.startswith('reports/') for x in z.namelist()))
    print('OUTPUT ZIP:',ZIP_PATH,'size GB:',round(ZIP_PATH.stat().st_size/1e9,3))
print('CSV/JSON reports at:',OUT)


## Step 8 — How to use the outputs
- Kaggle **Output → `dermalens_real_70_15_15.zip`**: download it or publish it as a new Kaggle dataset. It has three top-level image folders: `train`, `validation`, `test`.
- `dermalens_real_split/10_frozen_real_split_manifest.csv`: permanent source of truth for all real-only and mixed experiments. Put this and other **small report CSVs** in GitHub (not the ~3 GB images or 160 MB embeddings).
- `01_quality_and_sha256.csv`: corrupt/blank/small warnings.
- `04_phash_candidate_edges.csv`, `06_embedding_candidate_edges.csv`: similarity evidence.
- `08_group_summary.csv`, `09_large_groups_for_manual_review.csv`: review unusual group unions.
- `09_quarantined_cross_class_groups.csv`: disputed labels (do not train on these until reviewed).
- `11_per_class_split_counts.csv`, `11_group_counts_by_split_class.csv`, `12_pipeline_stage_counts.csv`: paper/report figures.

**Before training:** compare the saved `manifest` against the ZIP, review flagged groups and label correctness, retain a copy of exact split assignments, and use *the same* split in EfficientNet-B0, ConvNeXt V2 Tiny, DINOv2 ViT-B/14 and PanDerm Base. Real validation/test must stay isolated from synthetic training and near-duplicates. Human labels and source-specific copyright restrictions require a separate review.